## Summative Lab: Forest Fires Prevention

### Step 1: Load the Dataset

*   Install and import the ucimlrepo library.
*   Load the Forest Fires dataset:
 *   Predictors: Features from forest_fires.data.features.
 *   Target: forest_fires.data.targets.

In [ ]:
# Run pip install if necessary to access the UCI ML Repository (uncomment the next line)
! pip install ucimlrepo

In [ ]:
# Data
from ucimlrepo import fetch_ucirepo


forest_fires = fetch_ucirepo(id=162)
X = forest_fires.data.features
y = forest_fires.data.targets


# Display dataset structure
print(X.info())
print(X.describe())
print(y.head())

### Step 2: EDA

* Examine the dataset structure and summary statistics.
* Analyze correlations between predictors and the target variable.
* Plot scatterplots for key predictors vs. the target.
* Generate a residual plot to check for randomness in residuals.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

df = X.copy()
df["area"] = y

df.head()

In [ ]:
# check for missing values
print(df.isnull().sum())

In [ ]:
# analyze correlations between predictor and target variable
df_corr = df.copy()

df_corr["month"] = df_corr["month"].astype("category").cat.codes
df_corr["day"] = df_corr["day"].astype("category").cat.codes

corr = df_corr.corr(numeric_only=True)

plt.figure(figsize=(12,8))
sns.heatmap(corr,
            annot=True,
            cmap="coolwarm",
            fmt=".2f")
plt.show()

In [ ]:
#scatter plot
predictors = ["FFMC","DMC","DC","ISI","temp","RH","wind","rain"]

for col in predictors:
    plt.figure(figsize=(5,4))
    sns.scatterplot(data=df,
                    x=col,
                    y="area")
    plt.title(f"{col} vs Area")
    plt.show()
    

In [ ]:
#converting data types to numeric
X_numeric = pd.get_dummies(X, drop_first=True)

# Convert all columns to numeric
X_numeric = X_numeric.astype(float)

X_const = sm.add_constant(X_numeric)

# Make sure y is numeric
y_numeric = y.astype(float)

#convert to a data series
y_numeric = y["area"].astype(float)

#fit the model
model = sm.OLS(y_numeric, X_const).fit()
print(X_numeric.dtypes)

In [ ]:
#residual plot for randomness
import statsmodels.api as sm
#import pandas as pd

#X_numeric = pd.get_dummies(X, drop_first=True)
X_numeric = pd.get_dummies(X, drop_first=True, dtype=float)

X_const = sm.add_constant(X_numeric)


residuals = model.resid

plt.figure(figsize=(6,4))
plt.scatter(model.fittedvalues,
            residuals)

plt.axhline(0,color="red")
plt.xlabel("Fitted")
plt.ylabel("Residuals")
plt.title("Residual Plot")
plt.show()

### Step 3: Fit the regression models

* Fit a baseline multiple linear regression model with key predictors.
* Include nonlinear terms (e.g., quadratic transformations for significant predictors).
* Add interaction terms (e.g., between predictors with strong correlations).
* Incorporate indicator variables if categorical variables are present.
* Apply transformations (e.g., logarithmic transformations for skewed predictors).

In [ ]:
#baseline model
baseline_model = sm.OLS(y_numeric, X_const).fit()

print(baseline_model.summary())

In [ ]:
#quadratic transformations
X_quad = X_numeric.copy()

X_quad["temp_sq"] = X_quad["temp"]**2
X_quad["wind_sq"] = X_quad["wind"]**2

X_quad = sm.add_constant(X_quad)

quad_model = sm.OLS(y, X_quad).fit()

print(quad_model.summary())

In [ ]:
#interaction terms
X_inter = X_numeric.copy()

X_inter["temp_RH"] = X_inter["temp"] * X_inter["RH"]

X_inter = sm.add_constant(X_inter)

interaction_model = sm.OLS(y, X_inter).fit()

print(interaction_model.summary())

In [ ]:
#log transformation
import numpy as np
df["log_area"] = np.log1p(df["area"])

log_model = sm.OLS(df["log_area"],
                   X_const).fit()

print(log_model.summary())

### Step 4: Evaluate model diagnostics

* Compare models using metrics like 2R^2, adjusted RR^2, AIC, and BIC.
* Plot residuals and create Q-Q plots to assess normality.
* Identify influential observations using Cook's Distance.

In [ ]:
# comparison models
comparison = pd.DataFrame({

"Model":[
    "Baseline",
    "Quadratic",
    "Interaction",
    "Log Response"
],

"R2":[
baseline_model.rsquared,
quad_model.rsquared,
interaction_model.rsquared,
log_model.rsquared
],

"Adj R2":[
baseline_model.rsquared_adj,
quad_model.rsquared_adj,
interaction_model.rsquared_adj,
log_model.rsquared_adj
],

"AIC":[
baseline_model.aic,
quad_model.aic,
interaction_model.aic,
log_model.aic
],

"BIC":[
baseline_model.bic,
quad_model.bic,
interaction_model.bic,
log_model.bic
]

})

comparison

In [ ]:
#residual plot
plt.scatter(log_model.fittedvalues,
            log_model.resid)

plt.axhline(0,color="red")
plt.xlabel("Fitted")
plt.ylabel("Residuals")
plt.show()

In [ ]:
#Q-Q plot
sm.qqplot(log_model.resid,
          line='45')

plt.show()


In [ ]:
#cooks distance
influence = log_model.get_influence()

cooks = influence.cooks_distance[0]

plt.figure(figsize=(10,4))

plt.stem(cooks)

plt.title("Cook's Distance")

plt.show()

### Step 5: Apply regularization

* Use Ridge (L2) and Lasso (L1) regression from sklearn to handle multicollinearity.
* Extract coefficients and calculate Mean Squared Error (MSE).
* Compare the performance of Ridge and Lasso models.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error

X_ml = pd.get_dummies(X,
                      drop_first=True)

X_train,X_test,y_train,y_test = train_test_split(
    X_ml,
    y,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#ridge
ridge = Ridge(alpha=1)

ridge.fit(X_train_scaled,y_train)

ridge_pred = ridge.predict(X_test_scaled)

ridge_mse = mean_squared_error(y_test,
                               ridge_pred)

print("Ridge MSE:",ridge_mse)

print(ridge.coef_)

In [ ]:
#lasso
lasso = Lasso(alpha=0.1)

lasso.fit(X_train_scaled,y_train)

lasso_pred = lasso.predict(X_test_scaled)

lasso_mse = mean_squared_error(y_test,
                               lasso_pred)

print("Lasso MSE:",lasso_mse)

print(lasso.coef_)

In [ ]:
#comparison of ridge and lasso models
pd.DataFrame({

"Model":["Ridge","Lasso"],

"MSE":[ridge_mse,
       lasso_mse]

})

### Step 6: Prepare the data for binary classification

* Create a binary target variable based on a threshold in y (e.g., median or other percentile).
* Select relevant predictors and scale them using StandardScaler.

In [ ]:
#binary target is the median
median_area = df["area"].median()

df["fire_binary"] = (df["area"] > median_area).astype(int)

y_class = df["fire_binary"]

X_class = pd.get_dummies(X,
                         drop_first=True)

In [ ]:
#scale predictors
X_train,X_test,y_train,y_test = train_test_split(
    X_class,
    y_class,
    test_size=0.2,
    random_state=42
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

### Step 7: Train and evaluate a logistic regression model

Train a logistic regression model using the scaled predictors.

* Display coefficients and the intercept.
* Predict probabilities and binary outcomes.
* Evaluate performance using accuracy, confusion matrix, precision, recall, and F1-score.

In [ ]:
#logistic regression model
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000)

log_reg.fit(X_train_scaled,
            y_train)

In [ ]:
#coeeficients
print("Intercept")

print(log_reg.intercept_)

print("Coefficients")

coef = pd.DataFrame({

"Feature":X_class.columns,

"Coefficient":log_reg.coef_[0]

})

coef

In [ ]:
#prediction probabilities
probabilities = log_reg.predict_proba(X_test_scaled)

predictions = log_reg.predict(X_test_scaled)

In [ ]:
#accuracy,confusion matrix,precision,recall and F1 score
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

print("Accuracy:",
      accuracy_score(y_test,
                     predictions))

print("Precision:",
      precision_score(y_test,
                      predictions))

print("Recall:",
      recall_score(y_test,
                   predictions))

print("F1:",
      f1_score(y_test,
               predictions))

print(confusion_matrix(y_test,
                       predictions))

### Step 8: Check assumptions

* Use Variance Inflation Factor (VIF) to assess multicollinearity among predictors.

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Create dummy variables as integers (0/1), not booleans
X_vif = pd.get_dummies(X, drop_first=True)

# Convert bool columns to int
bool_cols = X_vif.select_dtypes(include='bool').columns
X_vif[bool_cols] = X_vif[bool_cols].astype(int)

# Convert everything to float
X_vif = X_vif.astype(np.float64)

# Add constant
X_vif = sm.add_constant(X_vif)

print(X_vif.values.dtype)   

In [ ]:
#convert the dataframe data to float type
X_vif = pd.DataFrame(
    X_vif.to_numpy(dtype=np.float64),
    columns=X_vif.columns
)
#
vif = pd.DataFrame({
    "Feature": X_vif.columns,
    "VIF": [
        variance_inflation_factor(X_vif.values, i)
        for i in range(X_vif.shape[1])
    ]
})
vif

### Step 9: Summative Findings

* Compare regression models and classification results.
* Highlight trade-offs between model simplicity, performance, and interpretability.
* Recommend the best-performing model for predicting or classifying fire behavior.

[Type your findings here.]

In [ ]:
#Using a linear regression model for prediction would be difficult due to the skewness in data for the burned area variable
#Applying a logarithmic transformation improved the regression assumptions by producing more normally distributed residuals.
#Quadratic and interaction terms slightly improved model performance, but the improvements were minimal.
#Among the regression models, the log-transformed model generally provided the best balance between goodness-of-fit and adherence to model assumptions.
#Ridge regression reduced coefficient variability caused by multicollinearity, while Lasso regression simplified the model by shrinking some coefficients to zero, 
# resulting in easier interpretation.
#Logistic regression successfully classified high-risk versus low-risk fire events after converting the response into a binary outcome. 
# Performance was evaluated using accuracy, precision, recall, F1-score, and the confusion matrix.
#Variance Inflation Factor (VIF) values indicated whether multicollinearity was present among predictors. Variables with high VIF values should be considered for removal or regularization.
#The log-transformed regression model is recommended for predicting burned area because it better satisfies regression assumptions, 
# while logistic regression is recommended when the objective is simply to classify whether a fire is likely to become large or remain small.